# GRN 5: optional task, DREAM5 cross-network transfer (exploratory)

This is the second-stage empirical check. Prepare two DREAM5 networks separately with Notebook 2 and save them as `dream5_net1_pair_data.npz` and `dream5_netX_pair_data.npz`. The two networks must use the same gene-index convention within each dataset, but gene identities need not match because the model uses pair expression profiles rather than gene IDs.

Cross-network transfer is **not automatically pure covariate shift**: both expression distributions and the regulatory mechanism/label mapping may differ across organisms. Report it as cross-network distribution shift, not as a controlled covariate-shift guarantee. Only run this notebook if a second matching gold-standard dataset is available.

In [ ]:
import os, sys
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive/', force_remount=True)
    PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/"
    if not os.path.isdir(PROJECT_DIR):
        PROJECT_DIR = os.getcwd()
except ImportError:
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)

from grn_experiment_utils import (
    split_indices, train_model, predict, evaluate,
    fit_temperature, apply_temperature, fit_isotonic, apply_isotonic, ece_score
)

data_dir=os.path.join(PROJECT_DIR,"data")
files=[f for f in os.listdir(data_dir) if f.startswith("dream5_") and f.endswith("_pair_data.npz")]

print("Available prepared DREAM5 datasets:",files)

if len(files)<2:
    raise FileNotFoundError("Prepare two distinct DREAM5 networks with Notebook 2 first. Do not use two copies of the same network.")


Mounted at /content/drive/
Available prepared DREAM5 datasets: ['dream5_net1_pair_data.npz', 'dream5_net3_pair_data.npz']


In [ ]:
# Pick source and target datasets explicitly. Update these names to match the files you prepared.
SOURCE_FILE=next((f for f in files if "net1" in f.lower()),files[0])
TARGET_FILE=next(f for f in files if f!=SOURCE_FILE)
src=np.load(os.path.join(data_dir,SOURCE_FILE),allow_pickle=True)
tgt=np.load(os.path.join(data_dir,TARGET_FILE),allow_pickle=True)
Xs,ys=src["X"].astype("float32"),src["y"].astype("float32")
Xt,yt=tgt["X"].astype("float32"),tgt["y"].astype("float32")
print("Source:",SOURCE_FILE,Xs.shape,"prevalence",ys.mean())
print("Target:",TARGET_FILE,Xt.shape,"prevalence",yt.mean())
if Xs.shape[1:]!=Xt.shape[1:]: raise ValueError("Source/target sequence shapes differ; use the same max_conditions setting.")


Source: dream5_net1_pair_data.npz (12000, 64, 2) prevalence 0.33308333
Target: dream5_net3_pair_data.npz (12000, 64, 2) prevalence 0.17216666


In [ ]:
def score_conditional_gap(y,p,bins=10,min_count=20):
    edges=np.linspace(0,1,bins+1); gaps=[]; weights=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        m=(p>=lo)&(p<(hi if hi<1 else hi+1e-9))
        if m.sum()>=min_count:
            gaps.append(abs(float(y[m].mean())-float(p[m].mean())))
            weights.append(float(m.mean()))
    return float(np.sum(np.asarray(gaps)*np.asarray(weights))) if gaps else float("nan")
# Train source-network models; fit post-hoc maps on source validation predictions only.
tr,va,te=split_indices(ys,seed=42)
rows=[]
for lam in [1.0,0.5,0.1]:
    model,hist=train_model(Xs,ys,tr,va,lam=lam,base_loss="bce",epochs=30,batch_size=32,
                           seed=42,patience=5)
    pv=predict(model,Xs[va]); T=fit_temperature(ys[va],pv)
    for dataset_name,X,y in [("source_test",Xs[te],ys[te]),("cross_network",Xt,yt)]:
        p=predict(model,X)
        row={"source":SOURCE_FILE,"evaluation":dataset_name,"lambda":lam,
             "method":"BCE" if lam==1 else "end_to_end",
             "prevalence":float(np.mean(y)),
             "marginal_calibration_in_large":float(abs(np.mean(p)-np.mean(y))),
             "score_conditional_calibration_gap":score_conditional_gap(y,p)}
        row.update(evaluate(y,p)); rows.append(row)
        if lam==1:
            row2={"source":SOURCE_FILE,"evaluation":dataset_name,"lambda":lam,
                  "method":"BCE+temperature","prevalence":float(np.mean(y)),
                  "marginal_calibration_in_large":float(abs(np.mean(apply_temperature(p,T))-np.mean(y))),
                  "score_conditional_calibration_gap":score_conditional_gap(y,apply_temperature(p,T))}
            row2.update(evaluate(y,apply_temperature(p,T))); rows.append(row2)
result=pd.DataFrame(rows); display(result)
result.to_csv(os.path.join(data_dir,"dream5_cross_network_metrics.csv"),index=False)
result.to_excel(os.path.join(data_dir,"dream5_cross_network_metrics.xlsx"),index=False)

/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/grn_experiment_utils.py:82: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder=nn.TransformerEncoder(enc_layer, num_layers=layers)
/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/grn_experiment_utils.py:82: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder=nn.TransformerEncoder(enc_layer, num_layers=layers)
/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/grn_experiment_utils.py:82: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder=nn.TransformerEncoder(enc_layer, num_layers=layers)


,source,evaluation,lambda,method,prevalence,marginal_calibration_in_large,score_conditional_calibration_gap,AUROC,AUPR,Brier,ECE10
0,dream5_net1_pair_data.npz,source_test,1.0,BCE,0.332778,0.048007,0.048007,0.833503,0.745715,0.151475,0.048007
1,dream5_net1_pair_data.npz,source_test,1.0,BCE+temperature,0.332778,0.035152,0.040954,0.833503,0.745715,0.150945,0.040954
2,dream5_net1_pair_data.npz,cross_network,1.0,BCE,0.172167,0.083377,0.152124,0.605996,0.251175,0.182806,0.152124
3,dream5_net1_pair_data.npz,cross_network,1.0,BCE+temperature,0.172167,0.097750,0.148760,0.605996,0.251175,0.180292,0.148760
4,dream5_net1_pair_data.npz,source_test,0.5,end_to_end,0.332778,0.006155,0.039969,0.815873,0.729356,0.155309,0.039969
5,dream5_net1_pair_data.npz,cross_network,0.5,end_to_end,0.172167,0.168209,0.183222,0.618944,0.263923,0.200310,0.183222
6,dream5_net1_pair_data.npz,source_test,0.1,end_to_end,0.332778,0.013316,0.029107,0.803911,0.705514,0.162603,0.029107
7,dream5_net1_pair_data.npz,cross_network,0.1,end_to_end,0.172167,0.162217,0.176244,0.651723,0.309629,0.197973,0.176244
